# RT-MLIDS — 02: Feature Analysis (MIG + SHAP)

Feature selection via Mutual Information Gain and post-hoc SHAP interpretability analysis.

**Results from paper experiment** (`rt_mlids_experiment.py`, runtime: 1h 46m)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Mutual Information Gain — Feature Selection

MIG retains the **top-30 features** that maximise class separability:

$$I(X;Y) = \sum P(x,y) \log \frac{P(x,y)}{P(x)P(y)}$$

Reducing from 53 → 30 features cuts inference latency by ~18% with no statistically significant accuracy loss.

In [2]:
print(f"Features retained: 30 / 53  (reduction: 43.4%)")

Features retained: 30 / 53  (reduction: 43.4%)


## SHAP Feature Importance — XGBoost

**Actual results from experiment** (mean absolute SHAP values, XGBoost on 500 test samples):

In [3]:
shap_data = {
    'Rank':    [1, 2, 3, 4, 5],
    'Feature': ['src_bytes', 'dst_host_srv_count', 'count',
                'service', 'dst_host_same_src_port_rate'],
    'Mean |SHAP|': [4.345443, 1.377532, 1.236061, 0.880139, 0.808574],
    'Interpretation': [
        'Source data volume — dominant DoS indicator (3.15× larger than #2)',
        'Destination host service connection count — probe signature',
        'Connection frequency to same host in past 2s',
        'Targeted network service type',
        'Rate of connections from same source port'
    ]
}
df_shap = pd.DataFrame(shap_data)
display(df_shap)

Rank,Feature,Mean |SHAP|,Interpretation
1,src_bytes,4.345443,Source data volume — dominant DoS indicator (3.15× larger than #2)
2,dst_host_srv_count,1.377532,Destination host service connection count — probe signature
3,count,1.236061,Connection frequency to same host in past 2s
4,service,0.880139,Targeted network service type
5,dst_host_same_src_port_rate,0.808574,Rate of connections from same source port


In [4]:
fig, ax = plt.subplots(figsize=(10, 5))

features   = df_shap['Feature'].tolist()
shap_vals  = df_shap['Mean |SHAP|'].tolist()
colors     = ['#E53935' if i == 0 else '#1E88E5' for i in range(len(features))]

bars = ax.barh(features[::-1], shap_vals[::-1], color=colors[::-1], edgecolor='white')
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax.set_title('Top-5 Features by SHAP Importance (XGBoost)', fontsize=13, fontweight='bold')

for bar, val in zip(bars, shap_vals[::-1]):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)

red_patch  = mpatches.Patch(color='#E53935', label='Dominant feature (src_bytes)')
blue_patch = mpatches.Patch(color='#1E88E5', label='Other top features')
ax.legend(handles=[red_patch, blue_patch])
ax.set_xlim(0, 5.2)

plt.tight_layout()
plt.savefig('../docs/fig_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Insight

`src_bytes` (SHAP = 4.345) is **3.15× more important** than the second-ranked feature. This is consistent with domain knowledge: DoS attacks flood the network with high-volume traffic from the source, making source byte count the single most discriminative feature for distinguishing attack from normal traffic.

The remaining top features — connection counts, service type, same-port rates — align with the signature patterns of **Probe** and **R2L** attacks, providing analyst-level transparency into model decisions.